# CPT Notebook B — Train (Section 6.1)

**Chạy lại notebook này mỗi session Kaggle (Run All), không sửa gì giữa các session.** Mỗi lần chạy nó tự: kéo checkpoint mới nhất từ Hub → train tiếp → push checkpoint đầy đủ mỗi `save_steps` → tự dừng trước khi hết giờ session. Lặp đến khi `global_step >= 6000` thì cell cuối tự merge LoRA và push bản final.

3 repo trên Hub (đừng nhầm):
- `{HF_USER}/vi-cpt-corpus-2048` — data đã pack (từ Notebook A, chạy trước 1 lần)
- `{HF_USER}/Qwen3-1.7B-vi-cpt-ckpt` — checkpoint để resume (adapter + optimizer + scheduler)
- `{HF_USER}/Qwen3-1.7B-vi-cpt` — bản merged cuối cùng, input cho Bước 2 (SFT)

**Trước khi chạy:** bật GPU T4 x1; secret `HF_TOKEN` (**Write** token) đã tạo. Username HF tự lấy từ token, không cần sửa gì trong notebook.

In [ ]:
!pip install -q -U unsloth

In [ ]:
# Config + login — username TỰ LẤY từ token (whoami), không gõ tay để khỏi sai namespace
import os, time
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download, whoami
login(UserSecretsClient().get_secret("HF_TOKEN"))

info = whoami()
HF_USER = info["name"]
role = ((info.get("auth") or {}).get("accessToken") or {}).get("role")
assert role != "read", "HF_TOKEN là READ token — tạo token WRITE tại hf.co/settings/tokens rồi cập nhật Kaggle Secret"
print(f"HF user: {HF_USER} (token: {role})")

DATA_REPO  = f"{HF_USER}/vi-cpt-corpus-2048"
CKPT_REPO  = f"{HF_USER}/Qwen3-1.7B-vi-cpt-ckpt"   # checkpoint resume
FINAL_REPO = f"{HF_USER}/Qwen3-1.7B-vi-cpt"        # CHỈ push khi đạt MAX_STEPS
OUT_DIR    = "/kaggle/working/cpt-checkpoints"
MAX_STEPS  = 6000    # ~400M token (32 seq x 2048 token/step)
BUDGET_H   = 8.0     # tự dừng sau 8h để kịp push trước khi session bị cắt (~9h)

In [ ]:
# Kéo checkpoint mới nhất từ Hub về (None nếu là session đầu tiên)
api = HfApi()
api.create_repo(CKPT_REPO, private=True, exist_ok=True)
steps = sorted(int(f.split("/")[0].split("-")[1]) for f in api.list_repo_files(CKPT_REPO)
               if f.startswith("checkpoint-"))
last_ckpt = None
if steps:
    name = f"checkpoint-{steps[-1]}"
    snapshot_download(CKPT_REPO, allow_patterns=f"{name}/*", local_dir=OUT_DIR)
    last_ckpt = os.path.join(OUT_DIR, name)
print("Resume từ:", last_ckpt or "(bắt đầu mới)")

In [ ]:
# Base model 4-bit + QLoRA r=64 (cấu hình cố định của Bước 1 — xem bảng 6.1 trong spec)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3-1.7B-Base", max_seq_length=2048, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=64, lora_alpha=64, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","up_proj","down_proj","o_proj","gate_proj"],
    use_rslora=True, use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
# Callback: mỗi lần save, upload NGUYÊN thư mục checkpoint lên Hub
# (phải là cả folder — optimizer.pt/scheduler.pt/rng_state/trainer_state.json — thì session sau
#  mới resume đúng; push_to_hub=True của TrainingArguments chỉ đẩy adapter nên KHÔNG dùng)
# và tự dừng khi vượt ngân sách giờ của session.
from transformers import TrainerCallback
class KaggleSessionCallback(TrainerCallback):
    def __init__(self): self.t0 = time.time()
    def on_save(self, args, state, control, **kw):
        name = f"checkpoint-{state.global_step}"
        api.upload_folder(repo_id=CKPT_REPO, folder_path=os.path.join(args.output_dir, name),
                          path_in_repo=name)
        print(f"Đã push {name} lên {CKPT_REPO}")
        if time.time() - self.t0 > BUDGET_H * 3600:
            control.should_training_stop = True
            print("Hết ngân sách giờ session — dừng an toàn, session sau tự resume.")

In [ ]:
# Train — data đã pack sẵn từ Notebook A, không tokenize lại
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
data = load_dataset(DATA_REPO)

trainer = Trainer(
    model=model,
    train_dataset=data["train"],
    eval_dataset={"vi": data["eval_vi"], "en": data["eval_en"]},  # eval_loss en tăng mạnh = forgetting
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[KaggleSessionCallback()],
    args=TrainingArguments(
        output_dir=OUT_DIR,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=1.5e-5, lr_scheduler_type="cosine", warmup_ratio=0.1,
        max_steps=MAX_STEPS,
        bf16=True, optim="paged_adamw_8bit",
        save_strategy="steps", save_steps=200, save_total_limit=2,
        eval_strategy="steps", eval_steps=600,
        logging_steps=20, report_to="none",
    ),
)
trainer.train(resume_from_checkpoint=last_ckpt)  # None -> train mới; có path -> nối tiếp đúng step
print(f"global_step = {trainer.state.global_step} / {MAX_STEPS}")

In [ ]:
# Khi đạt MAX_STEPS: merge LoRA vào base, push bản final — Bước 2 (SFT) load thẳng repo này
if trainer.state.global_step >= MAX_STEPS:
    model.push_to_hub_merged(FINAL_REPO, tokenizer, save_method="merged_16bit")
    print(f"CPT HOÀN TẤT — bản merged tại {FINAL_REPO}. Cập nhật README (bước 1 → done).")
else:
    print("Chưa đạt MAX_STEPS — mở lại notebook này ở session sau, nó tự resume.")

In [ ]:
# Smoke test nhanh (chỉ có ý nghĩa khi đã train kha khá / hoàn tất):
# model nên sinh tiếng Việt mạch lạc hơn base gốc — xem thêm tiêu chí 6.9 trong spec
FastLanguageModel.for_inference(model)
inputs = tokenizer("Việt Nam là một quốc gia", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))